# AI-sfa 3D 올리기 모델 학습 — Kaggle GPU

설정은 P1·P2 노트북과 같다: **GPU T4 x2(또는 P100)**, **Internet On**, Secrets `RCLONE_GDRIVE_AUTH_B64` 연결.
**Save Version → Save & Run All (Commit)**. 12시간에 멈추면 다시 실행하면 이어서 학습한다.

In [ ]:
# 1. 준비: rclone 설치·설정(인증값은 출력하지 않는다), 라이브러리
import os, json, base64, subprocess, pathlib
from kaggle_secrets import UserSecretsClient
raw = UserSecretsClient().get_secret('RCLONE_GDRIVE_AUTH_B64').strip()
d = json.loads(base64.b64decode(raw + '=' * (-len(raw) % 4)))
p = pathlib.Path.home() / '.config/rclone'; p.mkdir(parents=True, exist_ok=True)
lines = ['[gdrive]', 'type = drive', 'scope = drive.readonly,drive.file']
if d.get('client_id'): lines.append('client_id = ' + d['client_id'])
if d.get('client_secret'): lines.append('client_secret = ' + d['client_secret'])
lines.append('token = ' + d['token'])
(p / 'rclone.conf').write_text('\n'.join(lines) + '\n'); os.chmod(p / 'rclone.conf', 0o600)
del raw, d, lines
!cd /tmp && curl -sSLo rc.zip https://downloads.rclone.org/rclone-current-linux-amd64.zip && unzip -qo rc.zip && cp rclone-*/rclone /usr/local/bin/
!rclone lsd gdrive:AI-sfa 2>/dev/null | head -3
import torch; print('GPU 수:', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# 2. 자료·학습 코드 준비
import os, subprocess, time
os.makedirs('/kaggle/working/mocap', exist_ok=True); os.makedirs('/kaggle/working/out', exist_ok=True)
subprocess.run(['rclone', 'copy', 'gdrive:AI-sfa/02_분석자료/모캡_통합', '/kaggle/working/mocap'], check=True)
subprocess.run(['rclone', 'copy', 'gdrive:AI-sfa/02_분석자료/모델_3D올리기', '/kaggle/working/out'])
print(os.listdir('/kaggle/working/mocap'), os.listdir('/kaggle/working/out'))
open('/kaggle/working/lift_train.py', 'w', encoding='utf-8').write("\"\"\"2D→3D 올리기 모델 학습(GPU). 통합 모캡(tools/mocap_unify.py)을 매 단계 무작위 카메라로 투영해 학습 쌍을 만든다.\n\n사용: python tools/lift_train.py <mocap_unified 폴더> <출력 폴더> [--steps 60000] [--batch 256] [--win 81] [--test]\n입력 2D: RTMW 몸 17점(COCO 순서) [창, 17, 3] = (x, y, 보임). 골반 중심·몸 크기 1로 정규화. 빠진 점은 (0, 0, 0).\n출력 3D: 공통 18관절, 카메라 좌표(미터, 골반 기준). 창 전체 프레임을 한 번에 낸다.\n애니 대응 증강:\n- 멈춤: 창을 2·3프레임씩 같은 포즈로 유지(애니의 2컷·3컷). 정답 3D도 같이 유지.\n- 빠진 점: 관절별 무작위 제거, 하체 통째 제거(화면 밖), 손목 제거.\n- 잡음: 2D 가우스 잡음(몸 크기의 0~4%), 가끔 큰 튐.\n- 카메라: 방위 0~360°, 높이각 -25~50°, 거리 2~9 m, 초점 0.7~3.0.\n검증: 클립 번호로 나눈 5% 클립, 고정 시드. 지표 MPJPE(mm), 골반 정렬 후.\n걷기·감정 표현만 있는 클립은 덜 뽑고, 공격·피격·점프·구르기는 더 뽑는다(mocap_index 태그).\n학습 결과(가중치)는 Drive에 두고 저장소에는 넣지 않는다.\n\"\"\"\nimport argparse\nimport csv\nimport math\nimport os\nimport time\nimport zlib\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\n\nJN = [\"pelvis\", \"spine\", \"neck\", \"head\", \"l_shoulder\", \"r_shoulder\", \"l_elbow\", \"r_elbow\", \"l_wrist\", \"r_wrist\",\n      \"l_hip\", \"r_hip\", \"l_knee\", \"r_knee\", \"l_ankle\", \"r_ankle\", \"l_toe\", \"r_toe\"]\nJ = {n: i for i, n in enumerate(JN)}\nPAR = [-1, 0, 1, 2, 2, 2, 4, 5, 6, 7, 0, 0, 10, 11, 12, 13, 14, 15]\nSOURCES = [\"cmu\", \"lafan1\", \"bandai\", \"amass\"]\nBOOST = (\"공격\", \"피격·넘어짐\", \"점프·공중\", \"구르기·바닥\", \"방어·회피\")\nBODY_ORDER = [\"l_shoulder\", \"r_shoulder\", \"l_elbow\", \"r_elbow\", \"l_wrist\", \"r_wrist\", \"l_hip\", \"r_hip\", \"l_knee\",\n              \"r_knee\", \"l_ankle\", \"r_ankle\"]\n\n\ndef load(src_dir, index):\n    tags = {}\n    if index and os.path.exists(index):\n        tags = {(r[\"자료\"], r[\"클립\"]): r[\"태그\"] for r in csv.DictReader(open(index, encoding=\"utf-8\"))}\n    pos, clips = [], []  # clips: (전역 시작, 길이, 가중치, 검증 여부)\n    off = 0\n    for s in SOURCES:\n        p = os.path.join(src_dir, s + \".npz\")\n        if not os.path.exists(p):\n            continue\n        z = np.load(p)\n        pos.append(z[\"pos\"])\n        for ci, (st, ln, nm) in enumerate(zip(z[\"start\"], z[\"length\"], z[\"clip\"])):\n            t = tags.get((s, str(nm)), \"\")\n            w = 0.25 if t and set(t.split(\";\")) <= {\"걷기\", \"감정 표현\"} else (2.0 if any(b in t for b in BOOST) else 1.0)\n            val = zlib.crc32(f\"{s}:{ci}\".encode()) % 20 == 0  # 실행마다 같은 검증 클립\n            clips.append((off + int(st), int(ln), w, val))\n        off += len(z[\"pos\"])\n    return np.concatenate(pos), clips\n\n\ndef coco17(P):\n    \"\"\"[B,T,18,3] → [B,T,17,3]. 코·눈·귀는 머리 주변 근사점(머리 방향 기준).\"\"\"\n    head, neck = P[..., J[\"head\"], :], P[..., J[\"neck\"], :]\n    up = head - neck\n    up = up / (up.norm(dim=-1, keepdim=True) + 1e-6)\n    side = P[..., J[\"l_shoulder\"], :] - P[..., J[\"r_shoulder\"], :]\n    side = side / (side.norm(dim=-1, keepdim=True) + 1e-6)\n    fwd = torch.cross(side, up, dim=-1)\n    fwd = fwd / (fwd.norm(dim=-1, keepdim=True) + 1e-6)\n    pts = [head + 0.09 * fwd + 0.02 * up,\n           head + 0.07 * fwd + 0.035 * side + 0.05 * up, head + 0.07 * fwd - 0.035 * side + 0.05 * up,\n           head + 0.075 * side + 0.03 * up, head - 0.075 * side + 0.03 * up]\n    return torch.cat([torch.stack(pts, -2), P[..., [J[n] for n in BODY_ORDER], :]], -2)\n\n\ndef cameras(B, g, dev):\n    az = torch.rand(B, generator=g, device=dev) * 2 * math.pi\n    el = torch.deg2rad(torch.rand(B, generator=g, device=dev) * 75 - 25)\n    dist = torch.rand(B, generator=g, device=dev) * 7 + 2\n    foc = torch.rand(B, generator=g, device=dev) * 2.3 + 0.7\n    c = torch.stack([torch.sin(az) * torch.cos(el), torch.sin(el), torch.cos(az) * torch.cos(el)], -1) * dist[:, None]\n    f = -c / c.norm(dim=-1, keepdim=True)\n    upv = torch.tensor([0.0, 1.0, 0.0], device=dev).expand_as(f)\n    r = torch.cross(f, upv, dim=-1)\n    r = r / (r.norm(dim=-1, keepdim=True) + 1e-6)\n    u = torch.cross(r, f, dim=-1)\n    return torch.stack([r, u, f], 1), c, foc  # R [B,3,3]\n\n\ndef make_batch(pos, idx, win, g, dev, train=True):\n    \"\"\"idx: 창 시작 전역 프레임 [B]. 반환 x2d [B,T,17,3], y3d [B,T,18,3].\"\"\"\n    B = len(idx)\n    ar = torch.arange(win, device=dev)\n    P = pos[(idx[:, None] + ar[None]).reshape(-1)].reshape(B, win, 18, 3).float()\n    if train:  # 멈춤(2·3컷): 창의 일부 표본에서 프레임을 k개씩 묶어 첫 프레임으로 유지\n        k = torch.randint(1, 4, (B,), generator=g, device=dev)\n        hold = (torch.rand(B, generator=g, device=dev) < 0.5)\n        src = (ar[None] // k[:, None]) * k[:, None]\n        src = torch.where(hold[:, None], src, ar[None].expand(B, -1))\n        P = torch.gather(P, 1, src[:, :, None, None].expand(-1, -1, 18, 3))\n    root = P[:, win // 2, J[\"pelvis\"]].clone()\n    root[:, 1] += (torch.rand(B, generator=g, device=dev) * 0.9 - 0.3)\n    R, c, foc = cameras(B, g, dev)\n    Pc = torch.einsum(\"bij,btkj->btki\", R, P - (root + c)[:, None, None])\n    ok = (Pc[..., 2] > 0.3).all(-1).all(-1)\n    C = coco17(Pc)\n    X = foc[:, None, None, None] * C[..., :2] / C[..., 2:3].clamp(min=0.3)\n    X[..., 1] = -X[..., 1]\n    hip = (X[:, :, 11] + X[:, :, 12]) / 2\n    sh = (X[:, :, 5] + X[:, :, 6]) / 2\n    s = (sh - hip).norm(dim=-1).median(dim=1).values.clamp(min=1e-3)\n    X = (X - hip[:, :, None]) / s[:, None, None, None]\n    vis = torch.ones(*X.shape[:3], 1, device=dev)\n    if train:\n        X = X + torch.randn(X.shape, generator=g, device=dev) * (torch.rand(B, 1, 1, 1, generator=g, device=dev) * 0.04)\n        spike = torch.rand(X.shape[:3], generator=g, device=dev) < 0.005\n        X = X + spike[..., None] * torch.randn(X.shape, generator=g, device=dev) * 0.3\n        drop = torch.rand(X.shape[:3], generator=g, device=dev) < torch.rand(B, 1, 1, generator=g, device=dev) * 0.2\n        legs = (torch.rand(B, generator=g, device=dev) < 0.1)[:, None, None] & (torch.arange(17, device=dev) >= 13)\n        vis = (~(drop | legs)).float()[..., None]\n        X = X * vis\n    Y = Pc - Pc[:, :, :1]\n    return torch.cat([X, vis], -1)[ok], Y[ok]\n\n\nclass Lifter(nn.Module):\n    def __init__(self, win, d=256, layers=6, heads=8):\n        super().__init__()\n        self.inp = nn.Linear(17 * 3, d)\n        self.pos = nn.Parameter(torch.zeros(1, win, d))\n        enc = nn.TransformerEncoderLayer(d, heads, d * 4, dropout=0.1, batch_first=True, norm_first=True)\n        self.enc = nn.TransformerEncoder(enc, layers)\n        self.out = nn.Linear(d, 18 * 3)\n\n    def forward(self, x):\n        B, T = x.shape[:2]\n        h = self.enc(self.inp(x.reshape(B, T, -1)) + self.pos[:, :T])\n        return self.out(h).reshape(B, T, 18, 3)\n\n\ndef losses(pred, y):\n    mpjpe = (pred - y).norm(dim=-1).mean()\n    vel = ((pred[:, 1:] - pred[:, :-1]) - (y[:, 1:] - y[:, :-1])).norm(dim=-1).mean()\n    bp = (pred[:, :, 1:] - pred[:, :, PAR[1:]]).norm(dim=-1)\n    by = (y[:, :, 1:] - y[:, :, PAR[1:]]).norm(dim=-1)\n    bone = (bp - by).abs().mean()\n    return mpjpe, vel, bone\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"src\")\n    ap.add_argument(\"out\")\n    ap.add_argument(\"--index\", default=\"data/mocap/mocap_index_2026-10-21.csv\")\n    ap.add_argument(\"--steps\", type=int, default=60000)\n    ap.add_argument(\"--batch\", type=int, default=256)\n    ap.add_argument(\"--win\", type=int, default=81)\n    ap.add_argument(\"--lr\", type=float, default=3e-4)\n    ap.add_argument(\"--test\", action=\"store_true\", help=\"CPU로 몇 단계만 돌려 확인\")\n    a = ap.parse_args()\n    dev = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    os.makedirs(a.out, exist_ok=True)\n    pos_np, clips = load(a.src, a.index)\n    pos = torch.from_numpy(pos_np.astype(np.float32)).to(dev)\n    win = a.win\n    tr = [(s, n, w) for s, n, w, v in clips if not v and n >= win]\n    va = [(s, n) for s, n, w, v in clips if v and n >= win]\n    w = np.array([wt * (n - win + 1) for s, n, wt in tr], np.float64)\n    w /= w.sum()\n    print(f\"프레임 {len(pos_np)} 학습 클립 {len(tr)} 검증 클립 {len(va)} 장치 {dev}\", flush=True)\n    rng = np.random.default_rng(0)\n\n    def sample(n):\n        ci = rng.choice(len(tr), n, p=w)\n        return torch.tensor([tr[i][0] + rng.integers(0, tr[i][1] - win + 1) for i in ci], device=dev)\n\n    vidx = torch.tensor([s + o for s, n in va for o in range(0, n - win + 1, 60)][:4096], device=dev)\n    gv = torch.Generator(device=dev).manual_seed(123)\n    vx, vy = make_batch(pos, vidx, win, gv, dev, train=False)\n    g = torch.Generator(device=dev).manual_seed(1)\n    model = Lifter(win).to(dev)\n    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=0.01)\n    steps = 3 if a.test else a.steps\n    sched = torch.optim.lr_scheduler.OneCycleLR(opt, a.lr, total_steps=steps, pct_start=0.05)\n    ck = os.path.join(a.out, \"lifter.pt\")\n    start = 0\n    if os.path.exists(ck) and not a.test:\n        st = torch.load(ck, map_location=dev)\n        model.load_state_dict(st[\"model\"])\n        opt.load_state_dict(st[\"opt\"])\n        sched.load_state_dict(st[\"sched\"])\n        start = st[\"step\"]\n        print(\"이어서\", start, flush=True)\n    bs = 4 if a.test else a.batch\n    t0 = time.time()\n    for step in range(start, steps):\n        model.train()\n        x, y = make_batch(pos, sample(bs), win, g, dev)\n        m, v, b = losses(model(x), y)\n        loss = m + 0.5 * v + 0.2 * b\n        opt.zero_grad()\n        loss.backward()\n        nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n        opt.step()\n        sched.step()\n        if (step + 1) % (1 if a.test else 1000) == 0 or step + 1 == steps:\n            model.eval()\n            with torch.no_grad():\n                errs = [losses(model(vx[i:i + 512]), vy[i:i + 512])[0].item() for i in range(0, len(vx), 512)]\n            print(f\"{step + 1} 학습 {m.item() * 1000:.0f}mm 검증 MPJPE {np.mean(errs) * 1000:.1f}mm \"\n                  f\"{(time.time() - t0) / 60:.1f}분\", flush=True)\n            if not a.test:\n                torch.save({\"model\": model.state_dict(), \"opt\": opt.state_dict(), \"sched\": sched.state_dict(),\n                            \"step\": step + 1, \"win\": win, \"joints\": JN}, ck + \".tmp\")\n                os.replace(ck + \".tmp\", ck)  # 복사 중 반쯤 쓴 파일을 읽지 않도록\n\n\nif __name__ == \"__main__\":\n    main()\n")

In [ ]:
# 3. 학습(중간 저장본을 10분마다 Drive에 올림)
idx = [f for f in os.listdir('/kaggle/working/mocap') if f.endswith('.csv')][0]
pr = subprocess.Popen(['python', '/kaggle/working/lift_train.py', '/kaggle/working/mocap', '/kaggle/working/out',
                       '--index', '/kaggle/working/mocap/' + idx], stdout=open('/kaggle/working/lift.log', 'w'),
                      stderr=subprocess.STDOUT)
t0 = time.time()
while pr.poll() is None and time.time() - t0 < 11.5 * 3600:
    time.sleep(600)
    subprocess.run(['rclone', 'copy', '/kaggle/working/out/lifter.pt', 'gdrive:AI-sfa/02_분석자료/모델_3D올리기'])
    subprocess.run(['rclone', 'copy', '/kaggle/working/lift.log', 'gdrive:AI-sfa/02_분석자료/모델_3D올리기'])
    print(round((time.time() - t0) / 60), '분 |', open('/kaggle/working/lift.log').read().splitlines()[-1:], flush=True)
pr.terminate()
subprocess.run(['rclone', 'copy', '/kaggle/working/out/lifter.pt', 'gdrive:AI-sfa/02_분석자료/모델_3D올리기'])
subprocess.run(['rclone', 'copy', '/kaggle/working/lift.log', 'gdrive:AI-sfa/02_분석자료/모델_3D올리기'])
print('종료')